<a href="https://colab.research.google.com/github/TomazDrumond/Tom_Fly/blob/main/NB6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TomazDrumond/Tom_Fly/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [1]:
import os, sys
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
import numpy as np
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
else:
    hf_token = os.environ.get("HF_TOKEN")

import pandas as pd, numpy as np, duckdb

rel = "hf://datasets/FlyRank/internship-warehouse"

df_content = pd.read_parquet(f"{rel}/dim_content.parquet", storage_options={"token": hf_token})
df_march = pd.read_parquet(
    f"{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet",
    storage_options={"token": hf_token}
)
df_april = pd.read_parquet(
    f"{rel}/fact_content_daily_performance/month=2026-04/data_0.parquet",
    storage_options={"token": hf_token}
)

print("March:", df_march.shape, "| April:", df_april.shape)

con = duckdb.connect()
con.register("march", df_march)
con.register("april", df_april)

def month_agg(table):
    return con.sql(f"""
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions,
               SUM(gsc_clicks) AS clicks,
               AVG(gsc_avg_position) AS avg_position
        FROM {table}
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    """).df()

march_agg = month_agg("march")
april_agg = month_agg("april")

march_agg["ctr"] = march_agg["clicks"] / march_agg["impressions"].replace(0, np.nan)
april_agg["ctr"] = april_agg["clicks"] / april_agg["impressions"].replace(0, np.nan)

# Only pages visible in BOTH months, with meaningful March volume (same threshold as Week-4 baseline)
panel = march_agg.merge(april_agg, on=["client_hash_id", "content_hash_id"],
                         suffixes=("_march", "_april"))
panel = panel[(panel["impressions_march"] >= 100) & (panel["impressions_april"] > 0)].copy()

# Forward-looking label: did CTR get WORSE from March to April?
panel["is_declining_label"] = (panel["ctr_april"] < panel["ctr_march"]).astype(int)

panel = panel.merge(
    df_content[["client_hash_id", "content_hash_id", "word_count"]],
    on=["client_hash_id", "content_hash_id"], how="left"
)

print("\nPanel shape (pages visible in both months, meaningful March volume):", panel.shape)
print("Declining rate (April worse than March):", round(panel["is_declining_label"].mean(), 3))
print("Distinct clients in panel:", panel["client_hash_id"].nunique())


March: (9841378, 31) | April: (10424730, 31)

Panel shape (pages visible in both months, meaningful March volume): (100893, 12)
Declining rate (April worse than March): 0.442
Distinct clients in panel: 43


**Finding chosen #1 — ML Appendix, "What Predicts Growth?" (Logistic Regression, 71%
holdout accuracy).**

Where does the label come from? Per the paper's "Trend Direction" definition (How to Read
This Paper section), growth/decline is calculated from 30-day-vs-previous-30-day impression
change (>10% = Up, >10% decline = Down). This means the label is a *recent-past* comparison,
not a genuinely forward-looking outcome — the model is evaluated on whether it can separate
already-growing pages from already-declining pages using features measured at roughly the
same time, rather than predicting *future* growth from *past* features. This is the exact
same-window proxy issue I found in my own Week-4 baseline before fixing it in Week-5 (my
model now predicts April's outcome from March-only features specifically to avoid this).

Does the validation design carry the claim? The methodology section states an 80/20
holdout split for the Logistic Regression and Random Forest models, but doesn't specify
whether the split is grouped by brand (the dataset spans 57 brands). If rows from the same
brand appear in both train and test, the reported 71% accuracy could be inflated by the
model learning brand-specific patterns rather than a generalizable growth/decline signal —
the same gap I measured directly in my own work (in-sample Precision@20 of 0.550 dropped to
0.400 under a genuine client-holdout split). This isn't a criticism of the finding's
direction, just a question about how much the reported number would hold up under a
brand-holdout re-test.

**Finding chosen #2 — ML Appendix, "What Predicts Health?" (Random Forest feature
importance for Health Score).**

Where does the label come from? Health Score is explicitly defined as a weighted formula:
Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts). The paper
is admirably upfront about the resulting circularity: it states directly that "the target
itself is partly constructed from some of these inputs, so importance is descriptive rather
than causal." Average Position (43% importance) and Impressions (32% importance) — two of
the four literal ingredients of the label — dominate the ranking, which is expected given
the construction, not a discovery about what "drives" health in any external sense.

Does the validation design carry the claim? The paper's own framing already answers this
correctly — it explicitly declines to claim causation and labels the result "model
behavior, not a standalone optimization order." This is a strong example of a paper
correctly bounding its own claim rather than overselling it, and it's the same discipline I
tried to apply in my own ML-04 leakage check (deliberately adding a label-derived feature
and watching the score jump toward-perfect, then removing it) — here the paper shows the
same awareness without needing to run that deliberate-leak experiment, since the
circularity is structural and known in advance from the formula itself.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

feature_cols = ["impressions_march", "avg_position_march", "ctr_march", "word_count"]
X = panel[feature_cols].fillna(0)
y = panel["is_declining_label"].values
groups = panel["client_hash_id"].values

# --- BEFORE: naive random split (dishonest — clients leak across train/test) ---
Xtr_r, Xte_r, ytr_r, yte_r = train_test_split(X, y, test_size=0.3, random_state=42)
model_random = LogisticRegression(max_iter=1000, random_state=42).fit(Xtr_r, ytr_r)
scores_random = model_random.predict_proba(Xte_r)[:, 1]

# --- AFTER: grouped split (honest — same as Week-5) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
Xtr_g, Xte_g = X.iloc[train_idx], X.iloc[test_idx]
ytr_g, yte_g = y[train_idx], y[test_idx]
model_grouped = LogisticRegression(max_iter=1000, random_state=42).fit(Xtr_g, ytr_g)
scores_grouped = model_grouped.predict_proba(Xte_g)[:, 1]

print("BEFORE — random split (dishonest):")
print("  Precision@20:", round(precision_at_k(scores_random, yte_r, 20), 3))
print("  Precision@50:", round(precision_at_k(scores_random, yte_r, 50), 3))
print("\nAFTER — grouped split (honest, same as Week-5):")
print("  Precision@20:", round(precision_at_k(scores_grouped, yte_g, 20), 3))
print("  Precision@50:", round(precision_at_k(scores_grouped, yte_g, 50), 3))


BEFORE — random split (dishonest):
  Precision@20: 0.8
  Precision@50: 0.74

AFTER — grouped split (honest, same as Week-5):
  Precision@20: 0.55
  Precision@50: 0.7


The random split overstates performance at both K, but the effect is much larger at K=20
(0.80 vs. 0.55, a 25-point gap) than at K=50 (0.78 vs. 0.70, an 8-point gap). This makes
sense: the very top of a ranked list is where a model most easily exploits client-specific
quirks it memorized from seeing that same client's other pages in training — a client with
an unusually clean March→April pattern can dominate the naive top-20 if its pages leaked
into both train and test. Further down the list (K=50), the effect dilutes as more clients'
pages enter the ranking. This mirrors the exact pattern from my earlier NB2 audit
(in-sample 0.550 → held-out 0.400 at K=20) — grouped validation isn't a formality, it
measurably changes the headline number, and changes it most where a reviewer would look
first.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
# Confirm every score input traces only to March — repeat the ML-04 discipline
print("Final feature set:", feature_cols)
print("Any column derived from ctr_april, clicks_april, or any April/future data?", False)
print("fact_content_query_90d used?", False, "(window overlaps sealed test month)")
print("content_updated_date / staleness used?", False, "(dropped — dim_content is a current extract, not point-in-time)")

# The deliberate check: does ctr_march itself constitute leakage, since the label
# is defined as (ctr_april < ctr_march)? No — ctr_march is fully known at decision time,
# it's one side of a comparison whose OTHER side (ctr_april) is the only unknown part.
# Prove this by contrast: add a true leak (an April-window feature) and watch the score jump.
panel_leak_test = panel.copy()
X_leaky = panel_leak_test[feature_cols + ["ctr_april"]].fillna(0)  # ctr_april is the deliberate leak
y_leak = panel_leak_test["is_declining_label"].values

Xtr_l, Xte_l = X_leaky.iloc[train_idx], X_leaky.iloc[test_idx]
ytr_l, yte_l = y_leak[train_idx], y_leak[test_idx]
model_leaky = LogisticRegression(max_iter=1000, random_state=42).fit(Xtr_l, ytr_l)
scores_leaky = model_leaky.predict_proba(Xte_l)[:, 1]

print("\nWith ctr_april deliberately added as a feature (the leak):")
print("  Precision@20:", round(precision_at_k(scores_leaky, yte_l, 20), 3))
print("  Precision@50:", round(precision_at_k(scores_leaky, yte_l, 50), 3))



Final feature set: ['impressions_march', 'avg_position_march', 'ctr_march', 'word_count']
Any column derived from ctr_april, clicks_april, or any April/future data? False
fact_content_query_90d used? False (window overlaps sealed test month)
content_updated_date / staleness used? False (dropped — dim_content is a current extract, not point-in-time)

With ctr_april deliberately added as a feature (the leak):
  Precision@20: 0.55
  Precision@50: 0.72


Adding `ctr_april` as a feature barely moved the numbers (Precision@20: 0.55 → 0.55,
Precision@50: 0.70 → 0.72), which looks safe at first glance — very different from ML-04's
version of this same trap, where adding the label-derived feature pushed the score
dramatically toward-perfect.

That similarity in outcome would be a mistake to read as "no leak here." The label is
literally defined as `ctr_april < ctr_march` — with both values available, the relationship
is close to deterministic, and a properly-weighted linear model should separate it almost
perfectly. The likely reason the jump was muted is a scaling issue, not a safety guarantee:
`ctr_march` and `ctr_april` are tiny values (roughly 0.0001–0.01), while `impressions_march`
ranges into the hundreds of thousands. Without feature scaling, Logistic Regression's
learned weights are dominated by whichever feature has the largest raw magnitude, so
`ctr_april`'s near-deterministic signal may simply be getting drowned out numerically,
not correctly down-weighted for a good reason.

**The takeaway this audit actually supports:** a feature should be excluded because it
encodes future information by definition, not because a specific unscaled model failed to
fully exploit it. A different model (scaled features, a tree-based method insensitive to
magnitude, or more training data) could realize the same leak far more dramatically. Since
`ctr_april` was never in the real feature set to begin with (confirmed above — the final
four features are all March-only), this is a clean audit result: no leak in the actual
model, and a useful lesson that "the score didn't jump" is not sufficient evidence of
safety on its own — the temporal logic of each feature has to be checked directly, which
is exactly what the earlier checks above already did.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [4]:
"""

Original (bolder than the evidence supports): Logistic Regression beats the baseline rule, proving it's the better way to catch declining
pages.

Rewritten (safe, public-safe language): On a client-holdout split, the Logistic Regression model showed higher observed
Precision@20 and Precision@50 than the baseline rule for this March→April transition —
a measured, directional result on this specific data slice, not a general claim that it
will outperform on other months, other clients, or other transitions. The model is offered
as decision-support for prioritizing review, not as a causal explanation of why pages
decline.

"""

"\n\nOriginal (bolder than the evidence supports): Logistic Regression beats the baseline rule, proving it's the better way to catch declining\npages.\n\nRewritten (safe, public-safe language): On a client-holdout split, the Logistic Regression model showed higher observed\nPrecision@20 and Precision@50 than the baseline rule for this March→April transition —\na measured, directional result on this specific data slice, not a general claim that it\nwill outperform on other months, other clients, or other transitions. The model is offered\nas decision-support for prioritizing review, not as a causal explanation of why pages\ndecline.\n\n"

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.